# BOFA
Pages are split as left/right (even/odd), TOC on pages 2-3

In [1]:
import pypdfium2

pdf_path = "bofa-deposit-agreements.pdf"

pdf = pypdfium2.PdfDocument(pdf_path)
total_pages = len(pdf)
print(f"Pages: {total_pages}")

Pages: 29


In [2]:
toc_sections = []
path_stack = []
SKIP_PAGES = 3

for bookmark in pdf.get_toc():
    level = bookmark.level
    title = bookmark.get_title()
    dest = bookmark.get_dest()
    page_index = dest.get_index() if dest else None
    if page_index is None or page_index < SKIP_PAGES:
        continue

    while len(path_stack) > level:
        path_stack.pop()
    if len(path_stack) == level:
        path_stack.append(title)
    else:
        path_stack = path_stack[:level] + [title]
    path = " > ".join(path_stack)

    toc_sections.append({
        "title": title,
        "path": path,
        "level": level,
        "page": page_index,
    })

toc_sections = sorted(toc_sections, key=lambda x: x["page"])
print(f"Found {len(toc_sections)} table of contents paths (ignoring first {SKIP_PAGES} pages)\n")

for s in toc_sections:
    print(f"Page {s['page'] + 1}, Path: {s['path']}")

Found 228 table of contents paths (ignoring first 3 pages)

Page 4, Path: Opening an Account 
Page 4, Path: Opening an Account  > Your Identifying Information and How We Use It 
Page 4, Path: Opening an Account  > Your Identifying Information and How We Use It  > Information Required for Personal and Non-Personal Accounts 
Page 4, Path: Opening an Account  > Your Identifying Information and How We Use It  > Information Required for Personal and Non-Personal Accounts  > Additional Information Required for Non-Personal Accounts 
Page 4, Path: Opening an Account  > Your Identifying Information and How We Use It  > Information Required for Personal and Non-Personal Accounts  > Use and Sharing of Information 
Page 4, Path: Opening an Account  > Your Identifying Information and How We Use It  > Information Required for Personal and Non-Personal Accounts  > Contacting you by phone and monitoring phone calls 
Page 4, Path: Opening an Account  > Your Identifying Information and How We Use It  >

In [3]:
target_page_idx =  4 
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = textpage.get_text_range()

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 4
----------------------------------------
  1: Page 2 Page 3
  2: Opening an Account 
  3: Your Identifying Information and How We Use It 
  4: When you open a deposit account with us, you give us information about yourself and confirm it is correct. We may rely on 
  5: that information until you notify us of a change and we have had a reasonable time to act on the new information. 
  6: Federal law, including the USA PATRIOT Act, requires that we gather, verify and record certain information to have reasonable 
  7: assurance of your identity. During this process we may contact you for additional information. If you’ve already made a 
  8: deposit to fund your account, you may not have access to your money until we complete the process. If we can’t verify your 
  9: identity to our satisfaction, we won’t open the account or we may close it if it has already been funded. 
 10: Information Required for Personal and Non-Personal Accounts 
 11: To open an account with us, you must 

In [4]:
import re

def clean_bofa(text):
    text = re.sub(r"\bPage \d+\b", "", text)
    text = re.sub(r"05/2026", "", text)
    return text

target_page_idx =  4 
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = clean_bofa(textpage.get_text_range())

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 4
----------------------------------------
  2: Opening an Account 
  3: Your Identifying Information and How We Use It 
  4: When you open a deposit account with us, you give us information about yourself and confirm it is correct. We may rely on 
  5: that information until you notify us of a change and we have had a reasonable time to act on the new information. 
  6: Federal law, including the USA PATRIOT Act, requires that we gather, verify and record certain information to have reasonable 
  7: assurance of your identity. During this process we may contact you for additional information. If you’ve already made a 
  8: deposit to fund your account, you may not have access to your money until we complete the process. If we can’t verify your 
  9: identity to our satisfaction, we won’t open the account or we may close it if it has already been funded. 
 10: Information Required for Personal and Non-Personal Accounts 
 11: To open an account with us, you must provide current and

In [5]:
pages = []
for i in range(total_pages):
    textpage = pdf[i].get_textpage()
    cleaned_text = clean_bofa(textpage.get_text_range())
    pages.append(cleaned_text)

In [6]:
chunks = []
metadatas = []
ids = []

success_count = 0
debug_count = 0
failed_cases = []

def build_flexible_pattern(text):
    """
    Combines normalization and flexible pattern building.
    Strips punctuation, quotes, and parentheses, then constructs a regex 
    that safely matches titles even if the PDF text has varied spacing, 
    dashes, or punctuation between the words.
    """
    if not text:
        return None
    
    # 1. Normalize: lowercase, strip punctuation/quotes
    t = text.lower()
    t = re.sub(r"[.,?;:'’\"“”‘’()]+", " ", t)

    
    words = t.split()
    if not words:
        return None
    
    # 2. Build flexible regex allowing punctuation/spaces between title words
    escaped_words = [re.escape(w) for w in words]
    connector = r"[\s,.;:\-_/'’\"“”‘’()]+"
    
    pattern_str = r"\b" + connector.join(escaped_words) + r"\b"
    return re.compile(pattern_str, re.IGNORECASE)

leaf_sections = []
for i, sec in enumerate(toc_sections):
    current_level = sec["level"]
    is_parent = False
    if i + 1 < len(toc_sections):
        if toc_sections[i + 1]["level"] > current_level:
            is_parent = True

    # Check for table of contents using the helper logic
    norm_check = build_flexible_pattern(sec["title"])
    sec_title_norm = sec["title"].lower()
    
    if not is_parent and "table of contents" not in sec_title_norm:
        leaf_sections.append((i, sec))

print(f"Found {len(leaf_sections)} valid leaf sections to process.\n" + "="*50)

# Slice text using detected leaf nodes
for idx, (original_toc_idx, section) in enumerate(leaf_sections):
    start_page_idx = section["page"] - 1
    title = section["title"].strip()
    path = section["path"]
    
    next_title = leaf_sections[idx + 1][1]["title"].strip() if idx + 1 < len(leaf_sections) else "End of Document"
    
    print(f"[{idx + 1}/{len(leaf_sections)}] Processing Section:")
    print(f"  • Looking for Start Title : '{title}'")
    print(f"  • Cutting Boundary Target : '{next_title}'")
    
    found = False
    section_text = ""
    actual_start_page = start_page_idx
    
    # 1. Search for the start anchor
    start_pattern = build_flexible_pattern(title)
    
    for p_idx in range(start_page_idx, total_pages):
        page_text = pages[p_idx]
        
        if start_pattern:
            match = start_pattern.search(page_text)
            if match:
                found = True
                actual_start_page = p_idx
                section_text = page_text[match.start():]
                
                # 2. Dynamic Lookahead: Keep adding subsequent pages until we find the NEXT header
                current_p = p_idx + 1
                next_pattern = build_flexible_pattern(next_title)
                
                while current_p < total_pages:
                    if next_pattern and next_pattern.search(section_text):
                        break  # Found the next header! Stop pulling pages.
                    
                    section_text += f"\n{pages[current_p]}"
                    current_p += 1
                break

    if found:
        print(f"  -> Found start anchor on Page {actual_start_page + 1}")
    else:
        print(f"  -> [WARNING] Start anchor not found. Fallback to target page {start_page_idx + 1}")
        section_text = pages[start_page_idx]
        actual_start_page = start_page_idx

    # 3. Cut text right before the NEXT section using the flexible pattern
    if idx + 1 < len(leaf_sections):
        next_pattern = build_flexible_pattern(next_title)
        if next_pattern:
            next_match = next_pattern.search(section_text)
            if next_match:
                section_text = section_text[:next_match.start()]
                print(f"  -> [SUCCESS] Cleanly cut right before next title boundary.")
                success_count += 1
            else:
                print(f"  -> [NOTE] Next title boundary not found.")
                debug_count += 1
                # Save details for the final summary report
                failed_cases.append({
                    "section_index": idx + 1,
                    "current_title": title,
                    "target_next_title": next_title,
                    "collected_chars": len(section_text)
                })

    clean_text = section_text.strip()
    if clean_text:
        chunks.append(clean_text)
        metadatas.append({
            "path": path,
            "section_title": title,
            "start_page": actual_start_page + 1
        })
        
        safe_id = "".join(c if c.isalnum() else "_" for c in title).lower()
        ids.append(f"chunk_{original_toc_idx}_{safe_id[:15]}")
    
    print("-" * 50)

# --- FINAL SUMMARY REPORT ---
print("\n" + "="*50)
print("CHUNKING EXECUTION SUMMARY")
print("="*50)
print(f"Total Chunks Generated : {len(chunks)}")
print(f"Successful Boundary Cuts: {success_count}")
print(f"Failed Cuts: {debug_count}")

if failed_cases:
    print("\n--- LIST OF DEBUG / FAILED BOUNDARY CASES ---")
    for case in failed_cases:
        print(f"• Section {case['section_index']}: '{case['current_title']}'")
        print(f"  Target Next Title Missing: '{case['target_next_title']}'")
        print(f"  Characters Collected     : {case['collected_chars']}")
print("="*50)

Found 193 valid leaf sections to process.
[1/193] Processing Section:
  • Looking for Start Title : 'Information Required for Personal and Non-Personal Accounts'
  • Cutting Boundary Target : 'Additional Information Required for Non-Personal Accounts'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[2/193] Processing Section:
  • Looking for Start Title : 'Additional Information Required for Non-Personal Accounts'
  • Cutting Boundary Target : 'Use and Sharing of Information'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[3/193] Processing Section:
  • Looking for Start Title : 'Use and Sharing of Information'
  • Cutting Boundary Target : 'Contacting you by phone and monitoring phone calls'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
------

In [7]:
target_chunk_idx = 5

if target_chunk_idx < len(chunks):
    print(f"Chunk {target_chunk_idx}")
    print("=" * 40)
    print("METADATA")
    print("-" * 40)
    for key, value in metadatas[target_chunk_idx].items():
        print(f"  {key}: {value}")
    print("\nTEXT CONTENT")
    print("-" * 40)
    print(chunks[target_chunk_idx])
    print("=" * 40)
    print(f"Character count: {len(chunks[target_chunk_idx])}")
else:
    print(f"Index {target_chunk_idx} is out of range. Total chunks available: {len(chunks)}")

Chunk 5
METADATA
----------------------------------------
  path: Opening an Account  > Your Identifying Information and How We Use It  > Information Required for Personal and Non-Personal Accounts  > Information about authorized signers on accounts held by businesses 
  section_title: Information about authorized signers on accounts held by businesses
  start_page: 4

TEXT CONTENT
----------------------------------------
Information about authorized signers on accounts held by businesses 
Anyone who is authorized as a signer on a business account may request that we not share information gathered for 
purposes of determining eligibility for credit, insurance or other services with other Bank of America affiliates and selected 
third parties. To make this request, you can call us at 888.341.5000 or notify us at bankofamerica.com/privacy.
Character count: 426


In [8]:
for i in range(len(chunks)):
    print(f"Chunk {i}: {len(chunks[i])} characters")

Chunk 0: 991 characters
Chunk 1: 482 characters
Chunk 2: 637 characters
Chunk 3: 518 characters
Chunk 4: 701 characters
Chunk 5: 426 characters
Chunk 6: 309 characters
Chunk 7: 2653 characters
Chunk 8: 2730 characters
Chunk 9: 1424 characters
Chunk 10: 1729 characters
Chunk 11: 1119 characters
Chunk 12: 1002 characters
Chunk 13: 3165 characters
Chunk 14: 891 characters
Chunk 15: 243 characters
Chunk 16: 2086 characters
Chunk 17: 1215 characters
Chunk 18: 1063 characters
Chunk 19: 595 characters
Chunk 20: 2935 characters
Chunk 21: 875 characters
Chunk 22: 15527 characters
Chunk 23: 2936 characters
Chunk 24: 1393 characters
Chunk 25: 4637 characters
Chunk 26: 10569 characters
Chunk 27: 1806 characters
Chunk 28: 1359 characters
Chunk 29: 1230 characters
Chunk 30: 1015 characters
Chunk 31: 745 characters
Chunk 32: 682 characters
Chunk 33: 1439 characters
Chunk 34: 343 characters
Chunk 35: 941 characters
Chunk 36: 1817 characters
Chunk 37: 153 characters
Chunk 38: 1013 characters
Chunk 39: 

In [9]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize ChromaDB and create a unique collection for this document
chroma_client = chromadb.PersistentClient(path="./chroma_db")

doc_name = "bofa_deposit_agreement"  # Replace with your dynamic document filename/identifier
collection_name = f"doc_{"".join(c if c.isalnum() else "_" for c in doc_name).lower()}"
collection = chroma_client.get_or_create_collection(name=collection_name)

print(f"Initialized unique ChromaDB collection: '{collection_name}'\n" + "="*50)

# 2. Configure the text splitter (500 chars, 100 overlap)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

final_documents = []
final_metadatas = []
final_ids = []

MAX_CHUNK_SIZE = 500  # Threshold: only sub-chunk if text exceeds this length

# 3. Process existing TOC chunks conditionally
for parent_idx, (section_text, parent_meta, parent_id) in enumerate(zip(chunks, metadatas, ids)):
    
    # Check if the section is too large and needs further splitting
    if len(section_text) > MAX_CHUNK_SIZE:
        splits = text_splitter.split_text(section_text)
        
        for sub_idx, split_text in enumerate(splits):
            sub_id = f"{parent_id}_sub_{sub_idx}"
            sub_meta = {
                "section_title": parent_meta.get("section_title", "Unknown"),
                "start_page": parent_meta.get("start_page", 1),
                "parent_chunk_id": parent_id,
                "is_sub_chunk": True,
                "sub_index": sub_idx
            }
            final_documents.append(split_text)
            final_metadatas.append(sub_meta)
            final_ids.append(sub_id)
    else:
        # If it's already small enough, keep the original TOC chunk whole
        parent_meta["is_sub_chunk"] = False
        final_documents.append(section_text)
        final_metadatas.append(parent_meta)
        final_ids.append(parent_id)

print(f"Processed {len(chunks)} TOC sections into {len(final_documents)} final storable chunks.")

# 4. Add everything to ChromaDB
if final_documents:
    collection.add(
        documents=final_documents,
        metadatas=final_metadatas,
        ids=final_ids
    )
    print(f"Successfully embedded and stored {len(final_documents)} chunks in ChromaDB!")
else:
    print("[WARNING] No documents found to store.")

Initialized unique ChromaDB collection: 'doc_bofa_deposit_agreement'
Processed 193 TOC sections into 851 final storable chunks.


/Users/stevenha/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 31.9MiB/s]


Successfully embedded and stored 851 chunks in ChromaDB!


# Chase
1-indexed, 30 pages, last two pages are "Privacy Notice", TOC pages 2-4, need to remove stamp at bottom of all pages

In [10]:
pdf_path = "chase-deposit-account-agreement.pdf"

pdf = pypdfium2.PdfDocument(pdf_path)
total_pages = len(pdf)
print(f"Pages: {total_pages}")

Pages: 32


In [11]:
toc_sections = []
path_stack = []
SKIP_PAGES = 4

for bookmark in pdf.get_toc():
    level = bookmark.level
    title = bookmark.get_title()
    dest = bookmark.get_dest()
    page_index = dest.get_index() if dest else None
    if page_index is None or page_index < SKIP_PAGES:
        continue

    while len(path_stack) > level:
        path_stack.pop()
    if len(path_stack) == level:
        path_stack.append(title)
    else:
        path_stack = path_stack[:level] + [title]
    path = " > ".join(path_stack)

    toc_sections.append({
        "title": title,
        "path": path,
        "level": level,
        "page": page_index,
    })

toc_sections = sorted(toc_sections, key=lambda x: x["page"])
print(f"Found {len(toc_sections)} table of contents paths (ignoring first {SKIP_PAGES} pages)\n")

for s in toc_sections:
    print(f"Page {s['page'] + 1}, Path: {s['path']}")

Found 31 table of contents paths (ignoring first 4 pages)

Page 5, Path: Deposit Account Agreement
Page 5, Path: Deposit Account Agreement > A. Personal Accounts
Page 7, Path: Deposit Account Agreement > A. Personal Accounts > B. Business Accounts
Page 8, Path: Deposit Account Agreement > A. Personal Accounts > B. Business Accounts > A. Adding Money to Your Account
Page 9, Path: Deposit Account Agreement > A. Personal Accounts > B. Business Accounts > B. Posting Order and Processing
Page 10, Path: Deposit Account Agreement > A. Personal Accounts > B. Business Accounts > C. Overdrafts, Fees and Overdraft Protection
Page 11, Path: Deposit Account Agreement > A. Personal Accounts > B. Business Accounts > D. Electronic Funds Transfer Service Terms; Payments, Deposits and Transfers You Make or Receive byElectronic Methods
Page 15, Path: Deposit Account Agreement > A. Personal Accounts > B. Business Accounts > E. Other Ways to Use Your Money
Page 18, Path: Deposit Account Agreement > A. Pers

In [12]:
target_page_idx =  5
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = textpage.get_text_range()

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 5
----------------------------------------
  1: DEPOSIT ACCOUNT AGREEMENT
  2: JPMorgan Chase Bank, N.A. Member FDIC
  3: © 2026 JPMorgan Chase & Co.
  4: Page 5 of 30
  5: Return to Table of Contents Effective 6/14/2026
  6: Deposit Account Agreement
  7: This agreement (“Agreement”) is the contract that governs your account.
  8: Whether you have a personal or business deposit account, this document is the basic agreement between you and us (JPMorgan Chase Bank, N.A. the “Bank,” or 
  9: “Chase”). By signing a signature card or submitting an account application, or by using any of our deposit account services, you and anyone else identified as an 
 10: owner of the account agree to the terms in this Agreement. Customers of some of our business groups, such as Corporate Banking, will get a different agreement 
 11: and their accounts will be governed by that agreement, not this one. If you have a product that is not a deposit account, such as a credit card, this Agreement 
 12: d

In [13]:
def clean_chase(text):
    text = re.sub(r"DEPOSIT ACCOUNT AGREEMENT", "", text)
    text = re.sub(r"JPMorgan Chase Bank, N.A. Member FDIC", "", text)
    text = re.sub(r"© 2026 JPMorgan Chase & Co.", "", text)
    text = re.sub(r"Page \d+ of \d+", "", text)
    text = re.sub(r"Return to Table of Contents", "", text)
    text = re.sub(r"Effective \d+/\d+/\d+", "", text)
    return text

target_page_idx =  5 
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = clean_chase(textpage.get_text_range())

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 5
----------------------------------------
  6: Deposit Account Agreement
  7: This agreement (“Agreement”) is the contract that governs your account.
  8: Whether you have a personal or business deposit account, this document is the basic agreement between you and us (JPMorgan Chase Bank, N.A. the “Bank,” or 
  9: “Chase”). By signing a signature card or submitting an account application, or by using any of our deposit account services, you and anyone else identified as an 
 10: owner of the account agree to the terms in this Agreement. Customers of some of our business groups, such as Corporate Banking, will get a different agreement 
 11: and their accounts will be governed by that agreement, not this one. If you have a product that is not a deposit account, such as a credit card, this Agreement 
 12: does not apply to that product. Also, other products or services, such as online banking or retirement accounts, may have additional agreements. In the event of 
 13: an irreconci

In [14]:
pages = []
for i in range(total_pages):
    textpage = pdf[i].get_textpage()
    cleaned_text = clean_chase(textpage.get_text_range())
    pages.append(cleaned_text)

In [15]:
chunks = []
metadatas = []
ids = []

success_count = 0
debug_count = 0
failed_cases = []

def build_flexible_pattern(text):
    """
    Combines normalization and flexible pattern building.
    Strips punctuation, quotes, and parentheses, then constructs a regex 
    that safely matches titles even if the PDF text has varied spacing, 
    dashes, or punctuation between the words.
    """
    if not text:
        return None
    
    # 1. Normalize: lowercase, strip punctuation/quotes
    t = text.lower()
    t = re.sub(r"[?'’\"“”‘’()]*", " ", t)

    
    words = t.split()
    if not words:
        return None
    
    # 2. Build flexible regex allowing punctuation/spaces between title words
    escaped_words = [re.escape(w) for w in words]
    connector = r"[\s,.;:\-_/'’\"“”‘’()]*"
    
    pattern_str = r"\b" + connector.join(escaped_words) + r"\b"
    return re.compile(pattern_str, re.IGNORECASE)

leaf_sections = []
for i, sec in enumerate(toc_sections):
    current_level = sec["level"]
    is_parent = False
    if i + 1 < len(toc_sections):
        if toc_sections[i + 1]["level"] > current_level:
            is_parent = True

    # Check for table of contents using the helper logic
    norm_check = build_flexible_pattern(sec["title"])
    sec_title_norm = sec["title"].lower()
    
    if not is_parent and "table of contents" not in sec_title_norm:
        leaf_sections.append((i, sec))

print(f"Found {len(leaf_sections)} valid leaf sections to process.\n" + "="*50)

# Slice text using detected leaf nodes
for idx, (original_toc_idx, section) in enumerate(leaf_sections):
    start_page_idx = section["page"] - 1
    title = section["title"].strip()
    path = section["path"]
    
    next_title = leaf_sections[idx + 1][1]["title"].strip() if idx + 1 < len(leaf_sections) else "End of Document"
    
    print(f"[{idx + 1}/{len(leaf_sections)}] Processing Section:")
    print(f"  • Looking for Start Title : '{title}'")
    print(f"  • Cutting Boundary Target : '{next_title}'")
    
    found = False
    section_text = ""
    actual_start_page = start_page_idx
    
    # 1. Search for the start anchor
    start_pattern = build_flexible_pattern(title)
    
    for p_idx in range(start_page_idx, total_pages):
        page_text = pages[p_idx]
        
        if start_pattern:
            match = start_pattern.search(page_text)
            if match:
                found = True
                actual_start_page = p_idx
                section_text = page_text[match.start():]
                
                # 2. Dynamic Lookahead: Keep adding subsequent pages until we find the NEXT header
                current_p = p_idx + 1
                next_pattern = build_flexible_pattern(next_title)
                
                while current_p < total_pages:
                    if next_pattern and next_pattern.search(section_text):
                        break  # Found the next header! Stop pulling pages.
                    
                    section_text += f"\n{pages[current_p]}"
                    current_p += 1
                break

    if found:
        print(f"  -> Found start anchor on Page {actual_start_page + 1}")
    else:
        print(f"  -> [WARNING] Start anchor not found. Fallback to target page {start_page_idx + 1}")
        section_text = pages[start_page_idx]
        actual_start_page = start_page_idx

    # 3. Cut text right before the NEXT section using the flexible pattern
    if idx + 1 < len(leaf_sections):
        next_pattern = build_flexible_pattern(next_title)
        if next_pattern:
            next_match = next_pattern.search(section_text)
            if next_match:
                section_text = section_text[:next_match.start()]
                print(f"  -> [SUCCESS] Cleanly cut right before next title boundary.")
                success_count += 1
            else:
                print(f"  -> [NOTE] Next title boundary not found.")
                debug_count += 1
                # Save details for the final summary report
                failed_cases.append({
                    "section_index": idx + 1,
                    "current_title": title,
                    "target_next_title": next_title,
                    "collected_chars": len(section_text)
                })

    clean_text = section_text.strip()
    if clean_text:
        chunks.append(clean_text)
        metadatas.append({
            "path": path,
            "section_title": title,
            "start_page": actual_start_page + 1
        })
        
        safe_id = "".join(c if c.isalnum() else "_" for c in title).lower()
        ids.append(f"chunk_{original_toc_idx}_{safe_id[:15]}")
    
    print("-" * 50)

# --- FINAL SUMMARY REPORT ---
print("\n" + "="*50)
print("CHUNKING EXECUTION SUMMARY")
print("="*50)
print(f"Total Chunks Generated : {len(chunks)}")
print(f"Successful Boundary Cuts: {success_count}")
print(f"Failed Cuts: {debug_count}")

if failed_cases:
    print("\n--- LIST OF DEBUG / FAILED BOUNDARY CASES ---")
    for case in failed_cases:
        print(f"• Section {case['section_index']}: '{case['current_title']}'")
        print(f"  Target Next Title Missing: '{case['target_next_title']}'")
        print(f"  Characters Collected     : {case['collected_chars']}")
print("="*50)

Found 30 valid leaf sections to process.
[1/30] Processing Section:
  • Looking for Start Title : 'A. Personal Accounts'
  • Cutting Boundary Target : 'B. Business Accounts'
  -> Found start anchor on Page 5
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[2/30] Processing Section:
  • Looking for Start Title : 'B. Business Accounts'
  • Cutting Boundary Target : 'A. Adding Money to Your Account'
  -> Found start anchor on Page 7
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[3/30] Processing Section:
  • Looking for Start Title : 'A. Adding Money to Your Account'
  • Cutting Boundary Target : 'B. Posting Order and Processing'
  -> Found start anchor on Page 8
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[4/30] Processing Section:
  • Looking for Start Title : 'B. Posting Order and Processing

In [16]:
target_chunk_idx = 22

if target_chunk_idx < len(chunks):
    print(f"Chunk {target_chunk_idx}")
    print("=" * 40)
    print("METADATA")
    print("-" * 40)
    for key, value in metadatas[target_chunk_idx].items():
        print(f"  {key}: {value}")
    print("\nTEXT CONTENT")
    print("-" * 40)
    print(chunks[target_chunk_idx])
    print("=" * 40)
    print(f"Character count: {len(chunks[target_chunk_idx])}")
else:
    print(f"Index {target_chunk_idx} is out of range. Total chunks available: {len(chunks)}")

Chunk 22
METADATA
----------------------------------------
  path: Deposit Account Agreement > A. Personal Accounts > B. Business Accounts > F. Account Alerts
  section_title: F. Account Alerts
  start_page: 22

TEXT CONTENT
----------------------------------------
F. Account Alerts
If you receive or otherwise use Account Alerts, you agree to the following terms. If you are enrolled with our Digital Platforms, the terms of the digital or other 
online service agreements control the terms of these services instead.
•We may use an email address we have in our records for you that you provide to us for these services so we can send you certain information about your 
account. You may be automatically enrolled to receive certain Account Alerts. To manage your Alerts preferences, use our Digital Platforms. We do not 
charge for Account Alerts.
• Delivery of alerts may be delayed for various reasons, including service outages; technology failures; and system capacity limitations. The alerts 

In [17]:
for i in range(len(chunks)):
    print(f"Chunk {i}: {len(chunks[i])} characters")

Chunk 0: 14174 characters
Chunk 1: 4027 characters
Chunk 2: 9769 characters
Chunk 3: 5082 characters
Chunk 4: 4282 characters
Chunk 5: 28244 characters
Chunk 6: 17682 characters
Chunk 7: 1729 characters
Chunk 8: 1250 characters
Chunk 9: 1245 characters
Chunk 10: 912 characters
Chunk 11: 513 characters
Chunk 12: 1252 characters
Chunk 13: 791 characters
Chunk 14: 536 characters
Chunk 15: 3078 characters
Chunk 16: 2483 characters
Chunk 17: 2085 characters
Chunk 18: 5602 characters
Chunk 19: 1098 characters
Chunk 20: 689 characters
Chunk 21: 1959 characters
Chunk 22: 10885 characters
Chunk 23: 2030 characters
Chunk 24: 1252 characters
Chunk 25: 3519 characters
Chunk 26: 1461 characters
Chunk 27: 1319 characters
Chunk 28: 4951 characters
Chunk 29: 3784 characters


In [18]:
# 1. Initialize ChromaDB and create a unique collection for this document
chroma_client = chromadb.PersistentClient(path="./chroma_db")

doc_name = "chase_deposit_agreement"  # Replace with your dynamic document filename/identifier
collection_name = f"doc_{"".join(c if c.isalnum() else "_" for c in doc_name).lower()}"
collection = chroma_client.get_or_create_collection(name=collection_name)

print(f"Initialized unique ChromaDB collection: '{collection_name}'\n" + "="*50)

# 2. Configure the text splitter (500 chars, 100 overlap)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

final_documents = []
final_metadatas = []
final_ids = []

MAX_CHUNK_SIZE = 500  # Threshold: only sub-chunk if text exceeds this length

# 3. Process existing TOC chunks conditionally
for parent_idx, (section_text, parent_meta, parent_id) in enumerate(zip(chunks, metadatas, ids)):
    
    # Check if the section is too large and needs further splitting
    if len(section_text) > MAX_CHUNK_SIZE:
        splits = text_splitter.split_text(section_text)
        
        for sub_idx, split_text in enumerate(splits):
            sub_id = f"{parent_id}_sub_{sub_idx}"
            sub_meta = {
                "section_title": parent_meta.get("section_title", "Unknown"),
                "start_page": parent_meta.get("start_page", 1),
                "parent_chunk_id": parent_id,
                "is_sub_chunk": True,
                "sub_index": sub_idx
            }
            final_documents.append(split_text)
            final_metadatas.append(sub_meta)
            final_ids.append(sub_id)
    else:
        # If it's already small enough, keep the original TOC chunk whole
        parent_meta["is_sub_chunk"] = False
        final_documents.append(section_text)
        final_metadatas.append(parent_meta)
        final_ids.append(parent_id)

print(f"Processed {len(chunks)} TOC sections into {len(final_documents)} final storable chunks.")

# 4. Add everything to ChromaDB
if final_documents:
    collection.add(
        documents=final_documents,
        metadatas=final_metadatas,
        ids=final_ids
    )
    print(f"Successfully embedded and stored {len(final_documents)} chunks in ChromaDB!")
else:
    print("[WARNING] No documents found to store.")

Initialized unique ChromaDB collection: 'doc_chase_deposit_agreement'
Processed 30 TOC sections into 352 final storable chunks.
Successfully embedded and stored 352 chunks in ChromaDB!


# U.S. Bank
0-indexed, TOC pages 2-3, text in two column format

In [19]:
pdf_path = "usbank-Deposit-Account-Agreement.pdf"

pdf = pypdfium2.PdfDocument(pdf_path)
total_pages = len(pdf)
print(f"Pages: {total_pages}")

Pages: 26


In [20]:
toc_sections = []
path_stack = []
SKIP_PAGES = 3

for bookmark in pdf.get_toc():
    level = bookmark.level
    title = bookmark.get_title()
    dest = bookmark.get_dest()
    page_index = dest.get_index() if dest else None
    if page_index is None or page_index < SKIP_PAGES:
        continue

    while len(path_stack) > level:
        path_stack.pop()
    if len(path_stack) == level:
        path_stack.append(title)
    else:
        path_stack = path_stack[:level] + [title]
    path = " > ".join(path_stack)

    toc_sections.append({
        "title": title,
        "path": path,
        "level": level,
        "page": page_index,
    })

toc_sections = sorted(toc_sections, key=lambda x: x["page"])
print(f"Found {len(toc_sections)} table of contents paths (ignoring first {SKIP_PAGES} pages)\n")

for s in toc_sections:
    print(f"Page {s['page'] + 1}, Path: {s['path']}")

Found 100 table of contents paths (ignoring first 3 pages)

Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS
Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS > THIS IS AN AGREEMENT
Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS > DEFINITIONS
Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS > PRIVACY
Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS > CELLULAR PHONE CONTACT POLICY
Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS > MONITORING AND RECORDING COMMUNICATIONS
Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS > WAIVERS AND PRECEDENTS
Page 4, Path: TERMS APPLICABLE TO ALL ACCOUNTS > APPLICABLE LAW
Page 5, Path: TERMS APPLICABLE TO ALL ACCOUNTS > CUSTOMER IDENTIFICATION PROGRAM NOTICE (USA PATRIOT ACT)
Page 5, Path: TERMS APPLICABLE TO ALL ACCOUNTS > OWNER’S AUTHORITY
Page 5, Path: TERMS APPLICABLE TO ALL ACCOUNTS > AUTHORIZED ACCESS AND POWER OF ATTORNEY
Page 5, Path: TERMS APPLICABLE TO ALL ACCOUNTS > YOU CANNOT TRANSFER AN ACCOUNT
Page 5, Path: TERMS APPLICABLE TO ALL ACCOUNTS > CHANGE IN 

In [21]:
target_page_idx =  10
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = textpage.get_text_range()

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 10
----------------------------------------
  1: 9
  2: You can choose to link up to three eligible U.S. Bank accounts to your 
  3: personal checking account for Overdraft Protection. Owner(s) who are 
  4: signers on the checking account must also be signers on the account(s) 
  5: linked for Overdraft Protection.
  6: Eligible accounts include:
  7: • U.S. Bank personal savings account or money market account
  8: • U.S. Bank Reserve Line of Credit
  9: • U.S. Bank, State Farm or Edward Jones consumer credit card
 10: • A secondary U.S.  Bank or Edward Jones personal checking 
 11: account
 12: • U.S. Bank Personal Line of Credit
 13: • U.S. Bank Home Equity Line of Credit
 14: • Other U.S. Bank Lines of Credit
 15: If you have linked eligible accounts, and the negative Available Balance 
 16: in your checking account is or would be overdrawn by $5.01 or more, 
 17: the advance amount will transfer in multiples of $50.00. If the negative 
 18: Available Balance is or would be $

In [22]:
def clean_usb(text):
    text = re.sub(r"^\s*\d{1,3}\s*\n", "", text) # Remove page numbers at the start of the text
    return text

target_page_idx =  4
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = clean_usb(textpage.get_text_range())

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 4
----------------------------------------
  1: TERMS APPLICABLE TO ALL ACCOUNTS
  2: THIS IS AN AGREEMENT
  3: Welcome to U.S. Bank and thank you for opening an account with us. 
  4: This Agreement provides the general rules that apply to the account(s) 
  5: you have with U.S. Bank (“us”) described herein. Additional rules will 
  6: be provided in: 
  7: 1. disclosures we give you when you open your account for 
  8: example our pricing information found in:
  9: • Consumer Pricing Information (U.S. Bank and State Farm) call 
 10: 800-872-2657 to request a copy, we accept relay calls 
 11: • Edward Jones Consumer Pricing Information (exclusive to 
 12: Edward Jones clients), call 855-681-5325 to request a copy, 
 13: we accept relay calls
 14: • Business Pricing Information
 15: • U.S. Bank Business Essentials® Pricing Information
 16: • disclosure(s) and other fee disclosures 
 17: All disclosures can be obtained by visiting a U.S. Bank branch;
 18: 2. disclosures that are ap

In [23]:
pages = []
for i in range(total_pages):
    textpage = pdf[i].get_textpage()
    cleaned_text = clean_usb(textpage.get_text_range())
    pages.append(cleaned_text)

In [24]:
chunks = []
metadatas = []
ids = []

success_count = 0
debug_count = 0
failed_cases = []

def normalize_text(text):
    if not text:
        return ""
    t = text.upper()
    t = re.sub(r"[.,?\'’\"“”‘’()]+", " ", t)
    return " ".join(t.split())

def find_header_position(page_text, title):
    norm_title = normalize_text(title)
    norm_page = normalize_text(page_text)
    if not norm_title or norm_title not in norm_page:
        return -1
    
    words = norm_title.split()
    if not words:
        return -1
        
    escaped_words = [re.escape(w) for w in words]
    pattern_str = r"\b" + r"[\s,.;:\-_/'’\"“”‘’()]*".join(escaped_words) + r"\b"
    match = re.search(pattern_str, page_text)
    
    return match.start() if match else -1

leaf_sections = []
for i, sec in enumerate(toc_sections):
    current_level = sec["level"]
    is_parent = False
    if i + 1 < len(toc_sections):
        if toc_sections[i + 1]["level"] > current_level:
            is_parent = True

    sec_title_norm = normalize_text(sec["title"])
    if not is_parent and "table of contents" not in sec_title_norm:
        leaf_sections.append((i, sec))

print(f"Found {len(leaf_sections)} valid leaf sections to process.\n" + "="*50)

# Slice text using detected leaf nodes
for idx, (original_toc_idx, section) in enumerate(leaf_sections):
    start_page_idx = section["page"] - 1
    title = section["title"].strip()
    path = section["path"]
    
    next_title = leaf_sections[idx + 1][1]["title"].strip() if idx + 1 < len(leaf_sections) else "End of Document"
    
    print(f"[{idx + 1}/{len(leaf_sections)}] Processing Section:")
    print(f"  • Looking for Start Title : '{title}'")
    print(f"  • Cutting Boundary Target : '{next_title}'")
    
    found = False
    section_text = ""
    actual_start_page = start_page_idx
    
    # 1. Search for the start anchor page by page
    for p_idx in range(start_page_idx, total_pages):
        page_text = pages[p_idx]
        match_idx = find_header_position(page_text, title)
        
        if match_idx != -1:
            found = True
            actual_start_page = p_idx
            section_text = page_text[match_idx:]
            
            # 2. Dynamic Lookahead: Keep adding subsequent pages until we find the NEXT header
            current_p = p_idx + 1
            while current_p < total_pages:
                # Check if the next header is present in our growing section text
                if find_header_position(section_text, next_title) != -1:
                    break  # Found the next header! Stop pulling pages.
                
                section_text += f"\n{pages[current_p]}"
                current_p += 1
            break

    if found:
        print(f"  -> Found start anchor on Page {actual_start_page + 1}")
    else:
        print(f"  -> [WARNING] Start anchor not found. Fallback to target page {start_page_idx + 1}")
        section_text = pages[start_page_idx]
        actual_start_page = start_page_idx

    # 3. Cut text right before the NEXT section
    if idx + 1 < len(leaf_sections):
        next_match_idx = find_header_position(section_text, next_title)
        if next_match_idx != -1:
            section_text = section_text[:next_match_idx]
            print(f"  -> [SUCCESS] Cleanly cut right before next title boundary.")
            success_count += 1
        else:
            print(f"  -> [NOTE] Next title boundary not found.")
            debug_count += 1
            failed_cases.append({
                "section_index": idx + 1,
                "current_title": title,
                "target_next_title": next_title,
                "collected_chars": len(section_text)
            })

    clean_text = section_text.strip()
    if clean_text:
        chunks.append(clean_text)
        metadatas.append({
            "path": path,
            "section_title": title,
            "start_page": actual_start_page + 1
        })
        
        safe_id = "".join(c if c.isalnum() else "_" for c in title).lower()
        ids.append(f"chunk_{original_toc_idx}_{safe_id[:15]}")
    
    print("-" * 50)

# --- FINAL SUMMARY REPORT ---
print("\n" + "="*50)
print("CHUNKING EXECUTION SUMMARY")
print("="*50)
print(f"Total Chunks Generated : {len(chunks)}")
print(f"Successful Boundary Cuts: {success_count}")
print(f"Failed Cuts (Debug)     : {debug_count}")

if failed_cases:
    print("\n--- LIST OF DEBUG / FAILED BOUNDARY CASES ---")
    for case in failed_cases:
        print(f"• Section {case['section_index']}: '{case['current_title']}'")
        print(f"  Target Next Title Missing: '{case['target_next_title']}'")
        print(f"  Characters Collected     : {case['collected_chars']}")
print("="*50)

Found 95 valid leaf sections to process.
[1/95] Processing Section:
  • Looking for Start Title : 'THIS IS AN AGREEMENT'
  • Cutting Boundary Target : 'DEFINITIONS'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[2/95] Processing Section:
  • Looking for Start Title : 'DEFINITIONS'
  • Cutting Boundary Target : 'PRIVACY'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[3/95] Processing Section:
  • Looking for Start Title : 'PRIVACY'
  • Cutting Boundary Target : 'CELLULAR PHONE CONTACT POLICY'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[4/95] Processing Section:
  • Looking for Start Title : 'CELLULAR PHONE CONTACT POLICY'
  • Cutting Boundary Target : 'MONITORING AND RECORDING COMMUNICATIO

In [25]:
target_chunk_idx = 32

if target_chunk_idx < len(chunks):
    print(f"Chunk {target_chunk_idx}")
    print("=" * 40)
    print("METADATA")
    print("-" * 40)
    for key, value in metadatas[target_chunk_idx].items():
        print(f"  {key}: {value}")
    print("\nTEXT CONTENT")
    print("-" * 40)
    print(chunks[target_chunk_idx])
    print("=" * 40)
    print(f"Character count: {len(chunks[target_chunk_idx])}")
else:
    print(f"Index {target_chunk_idx} is out of range. Total chunks available: {len(chunks)}")

Chunk 32
METADATA
----------------------------------------
  path: TERMS APPLICABLE TO ALL ACCOUNTS > STOP PAYMENTS
  section_title: STOP PAYMENTS
  start_page: 13

TEXT CONTENT
----------------------------------------
STOP PAYMENTS
Unless otherwise provided, the rules in this section cover stopping 
payment of items such as checks and drafts drawn on your account. 
Rules for stopping payment on other types of transfers of funds, such 
as consumer electronic fund transfers, are covered elsewhere in the 
Electronic Fund Transfers for Consumer Customers section.
Here is what you must do to stop a payment: 
ACH Check Conversion
An ACH Check Conversion is a transaction that starts as a paper check 
that you give to a merchant. The merchant converts the check, using 
the information on the paper, to send an electronic message to a bank 
to immediately take the money from your account. The merchant is 
required to post a notice about ACH Check Conversion in a prominent 
and conspicuous locat

In [26]:
for i in range(len(chunks)):
    print(f"Chunk {i}: {len(chunks[i])} characters")

Chunk 0: 2896 characters
Chunk 1: 2217 characters
Chunk 2: 1406 characters
Chunk 3: 683 characters
Chunk 4: 583 characters
Chunk 5: 405 characters
Chunk 6: 839 characters
Chunk 7: 1085 characters
Chunk 8: 1389 characters
Chunk 9: 3712 characters
Chunk 10: 203 characters
Chunk 11: 541 characters
Chunk 12: 810 characters
Chunk 13: 276 characters
Chunk 14: 910 characters
Chunk 15: 1750 characters
Chunk 16: 4925 characters
Chunk 17: 1344 characters
Chunk 18: 171 characters
Chunk 19: 3867 characters
Chunk 20: 1266 characters
Chunk 21: 1148 characters
Chunk 22: 4691 characters
Chunk 23: 6046 characters
Chunk 24: 7150 characters
Chunk 25: 7674 characters
Chunk 26: 11078 characters
Chunk 27: 1520 characters
Chunk 28: 4687 characters
Chunk 29: 15330 characters
Chunk 30: 626 characters
Chunk 31: 1015 characters
Chunk 32: 3823 characters
Chunk 33: 2003 characters
Chunk 34: 3810 characters
Chunk 35: 3772 characters
Chunk 36: 593 characters
Chunk 37: 1239 characters
Chunk 38: 1297 characters
Chunk 

In [27]:
# 1. Initialize ChromaDB and create a unique collection for this document
chroma_client = chromadb.PersistentClient(path="./chroma_db")

doc_name = "usbank_deposit_agreement"  # Replace with your dynamic document filename/identifier
collection_name = f"doc_{"".join(c if c.isalnum() else "_" for c in doc_name).lower()}"
collection = chroma_client.get_or_create_collection(name=collection_name)

print(f"Initialized unique ChromaDB collection: '{collection_name}'\n" + "="*50)

# 2. Configure the text splitter (500 chars, 100 overlap)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

final_documents = []
final_metadatas = []
final_ids = []

MAX_CHUNK_SIZE = 500  # Threshold: only sub-chunk if text exceeds this length

# 3. Process existing TOC chunks conditionally
for parent_idx, (section_text, parent_meta, parent_id) in enumerate(zip(chunks, metadatas, ids)):
    
    # Check if the section is too large and needs further splitting
    if len(section_text) > MAX_CHUNK_SIZE:
        splits = text_splitter.split_text(section_text)
        
        for sub_idx, split_text in enumerate(splits):
            sub_id = f"{parent_id}_sub_{sub_idx}"
            sub_meta = {
                "section_title": parent_meta.get("section_title", "Unknown"),
                "start_page": parent_meta.get("start_page", 1),
                "parent_chunk_id": parent_id,
                "is_sub_chunk": True,
                "sub_index": sub_idx
            }
            final_documents.append(split_text)
            final_metadatas.append(sub_meta)
            final_ids.append(sub_id)
    else:
        # If it's already small enough, keep the original TOC chunk whole
        parent_meta["is_sub_chunk"] = False
        final_documents.append(section_text)
        final_metadatas.append(parent_meta)
        final_ids.append(parent_id)

print(f"Processed {len(chunks)} TOC sections into {len(final_documents)} final storable chunks.")

# 4. Add everything to ChromaDB
if final_documents:
    collection.add(
        documents=final_documents,
        metadatas=final_metadatas,
        ids=final_ids
    )
    print(f"Successfully embedded and stored {len(final_documents)} chunks in ChromaDB!")
else:
    print("[WARNING] No documents found to store.")

Initialized unique ChromaDB collection: 'doc_usbank_deposit_agreement'
Processed 94 TOC sections into 607 final storable chunks.
Successfully embedded and stored 607 chunks in ChromaDB!


# Wells Fargo

1-indexed, TOC on page 3

In [28]:
pdf_path = "wells-fargo-agreement.pdf"

pdf = pypdfium2.PdfDocument(pdf_path)
total_pages = len(pdf)
print(f"Pages: {total_pages}")

Pages: 44


In [29]:
toc_sections = []
path_stack = []
SKIP_PAGES = 3

for bookmark in pdf.get_toc():
    level = bookmark.level
    title = bookmark.get_title()
    dest = bookmark.get_dest()
    page_index = dest.get_index() if dest else None
    if page_index is None or page_index < SKIP_PAGES:
        continue

    while len(path_stack) > level:
        path_stack.pop()
    if len(path_stack) == level:
        path_stack.append(title)
    else:
        path_stack = path_stack[:level] + [title]
    path = " > ".join(path_stack)

    toc_sections.append({
        "title": title,
        "path": path,
        "level": level,
        "page": page_index,
    })

toc_sections = sorted(toc_sections, key=lambda x: x["page"])
print(f"Found {len(toc_sections)} table of contents paths (ignoring first {SKIP_PAGES} pages)\n")

for s in toc_sections:
    print(f"Page {s['page'] + 1}, Path: {s['path']}")

Found 271 table of contents paths (ignoring first 3 pages)

Page 4, Path: Account Basics
Page 4, Path: Account Basics > Account Ownership
Page 4, Path: Account Basics > Account Ownership > Forms of account ownership:
Page 4, Path: Account Basics > Account Ownership > Forms of account ownership: > For joint accounts
Page 4, Path: Account Basics > Account Ownership > Forms of account ownership: > Joint tenants with right of survivorship:
Page 4, Path: Account Basics > Account Ownership > Pay On Death (POD) account
Page 4, Path: Account Basics > Account Ownership > Accounts established for minors
Page 4, Path: Account Basics > Account Ownership > Accounts established for minors > Uniform Transfers/Gifts to Minors Act account.
Page 4, Path: Account Basics > Account Ownership > Accounts established for minors > Minor by account.
Page 5, Path: Account Basics > Account Ownership > Transfer of account ownership
Page 5, Path: Account Basics > Account Ownership > When an owner does not sign acco

In [30]:
target_page_idx =  7
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = textpage.get_text_range()

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 7
----------------------------------------
  1: Deposit Account Agreement 7 
  2: Interest
  3: Interest-bearing 
  4: accounts 
  5: Calculating the applicable interest rate: When you open an interest-bearing account, we provide a 
  6: rate sheet listing the current interest rate and Annual Percentage Yield (APY) for your account. Interest￾bearing accounts earn interest at a variable rate, except CDs. The interest rate may be as low as 0.00%, 
  7: and we may change the interest rate for variable-rate accounts at any time. The interest rate may vary 
  8: depending on your daily balances (tiered-rate account). We may pay the same interest rate on more than 
  9: one tier. The tiers and corresponding interest rates are disclosed in the rate sheet. 
 10: We calculate interest using the daily collected balance method, applying a daily periodic rate to the 
 11: collected balance in your account each day. Interest is calculated using a 365-day year, unless otherwise 
 12: noted for 

In [31]:
def clean_fargo(text):
    text = re.sub(r"\d*\s*Deposit Account Agreement\s*\d*", "", text)
    return text

target_page_idx =  14
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = clean_fargo(textpage.get_text_range())

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 14
----------------------------------------
  1: Standard Overdraft 
  2: Coverage 
  3: The bank typically does not 
  4: pay overdrafts if your account 
  5: is overdrawn or you have had 
  6: excessive overdrafts.
  7: Step 3: If the available balance is not enough to pay all of your transactions, we:
  8:  • Use Overdraft Protection (if you have it) by transferring and/or advancing available funds from a 
  9: linked savings and/or credit account.
 10:  • Then, decide whether to pay your transactions presented to us for payment into overdraft, or 
 11: return them unpaid. Paying an item into overdraft means that we pay an item even though your 
 12: available balance is not sufficient to cover that item, resulting in your account having a negative 
 13: balance. Returning an item unpaid means that we do not pay your transaction. At our discretion, 
 14: we may pay a check, other transactions using your account and routing number, or automatic bill 
 15: payment into overdraft,

In [32]:
pages = []
for i in range(total_pages):
    textpage = pdf[i].get_textpage()
    cleaned_text = clean_fargo(textpage.get_text_range())
    pages.append(cleaned_text)

In [33]:
chunks = []
metadatas = []
ids = []

success_count = 0
debug_count = 0
failed_cases = []

def normalize_text(text):
    if not text:
        return ""
    t = text.lower()
    t = re.sub(r"[\u00ad\u200b\u200c\u200d\u20fe]+", "", t)
    t = re.sub(r"[.,?;:\-_/'’\"“”‘’()]+", " ", t)
    return " ".join(t.split())

def build_flexible_pattern(text, max_words=None):
    """
    Builds a flexible regex pattern. If max_words is set, it shortens 
    long sentence-like TOC titles to use just their opening words.
    """
    if not text:
        return None
    
    norm_text = normalize_text(text)
    words = norm_text.split()
    if not words:
        return None
    
    # If the title is an overly long sentence, take only the first N words
    if max_words and len(words) > max_words:
        words = words[:max_words]
    
    escaped_words = [re.escape(w) for w in words]
    connector = r"[\s,.;:\-_/'’\"“”‘’()]*"
    
    pattern_str = r"\b" + connector.join(escaped_words) + r"\b"
    return re.compile(pattern_str, re.IGNORECASE | re.UNICODE)

def find_header_position(page_text, title):
    norm_text = normalize_text(title)
    norm_words = norm_text.split()
    
    if not norm_words:
        return -1

    # 1. Try matching the full title first
    pattern = build_flexible_pattern(title)
    if pattern:
        match = pattern.search(page_text)
        if match:
            return match.start()
            
    # 2. Fallback 1: If title has more than 4 words, try matching the first 4 words
    if len(norm_words) >= 4:
        prefix_4 = " ".join(norm_words[:4])
        p4_pattern = build_flexible_pattern(prefix_4)
        if p4_pattern:
            match = p4_pattern.search(page_text)
            if match:
                return match.start()

    # 3. Fallback 2: If title is shorter or 4 words failed, try matching the first 3 words
    if len(norm_words) >= 3:
        prefix_3 = " ".join(norm_words[:3])
        p3_pattern = build_flexible_pattern(prefix_3)
        if p3_pattern:
            match = p3_pattern.search(page_text)
            if match:
                return match.start()
                
    # 4. Fallback 3: Core words fallback (ignoring stop words like 'the', 'and', 'to')
    core_words = [w for w in norm_words if w not in {"the", "a", "an", "to", "and", "of", "in", "for"}]
    if len(core_words) >= 2:
        core_pattern = build_flexible_pattern(" ".join(core_words[:3]))
        if core_pattern:
            match = core_pattern.search(page_text)
            if match:
                return match.start()
                
    return -1

# Filter explicitly for Level 2 sections
target_sections = []
for i, sec in enumerate(toc_sections):
    sec_title_norm = sec["title"].lower()
    if sec.get("level") == 2 and "table of contents" not in sec_title_norm:
        target_sections.append((i, sec))

print(f"Found {len(target_sections)} level 2 sections to process.\n" + "="*50)

# Slice text using detected level 2 nodes
for idx, (original_toc_idx, section) in enumerate(target_sections):
    start_page_idx = section["page"] - 1
    title = section["title"].strip()
    path = section["path"]
    
    next_title = target_sections[idx + 1][1]["title"].strip() if idx + 1 < len(target_sections) else "End of Document"
    
    print(f"[{idx + 1}/{len(target_sections)}] Processing Section:")
    print(f"  • Looking for Start Title : '{title}'")
    print(f"  • Cutting Boundary Target : '{next_title}'")
    
    found = False
    section_text = ""
    actual_start_page = start_page_idx
    
    # 1. Search for the start anchor
    for p_idx in range(start_page_idx, total_pages):
        page_text = pages[p_idx]
        match_idx = find_header_position(page_text, title)
        
        if match_idx != -1:
            found = True
            actual_start_page = p_idx
            section_text = page_text[match_idx:]
            
            # 2. Dynamic Lookahead with safety character limit (e.g., max 35k chars per chunk)
            current_p = p_idx + 1
            while current_p < total_pages:
                if find_header_position(section_text, next_title) != -1:
                    break  # Found next header!
                
                # Safety check: Prevent runaway chunks if next header doesn't exist in body
                if len(section_text) > 35000:
                    print(f"      [SAFETY] Section grew past 35k chars without finding next boundary. Stopping lookahead.")
                    break
                    
                section_text += f"\n{pages[current_p]}"
                current_p += 1
            break

    if found:
        print(f"  -> Found start anchor on Page {actual_start_page + 1}")
    else:
        print(f"  -> [WARNING] Start anchor not found. Fallback to target page {start_page_idx + 1}")
        section_text = pages[start_page_idx]
        actual_start_page = start_page_idx

    # 3. Cut text right before the NEXT section
    if idx + 1 < len(target_sections):
        next_match_idx = find_header_position(section_text, next_title)
        if next_match_idx != -1:
            section_text = section_text[:next_match_idx]
            print(f"  -> [SUCCESS] Cleanly cut right before next title boundary.")
            success_count += 1
        else:
            print(f"  -> [NOTE] Next title boundary not found.")
            debug_count += 1
            failed_cases.append({
                "section_index": idx + 1,
                "current_title": title,
                "target_next_title": next_title,
                "collected_chars": len(section_text)
            })

    clean_text = section_text.strip()
    if clean_text:
        chunks.append(clean_text)
        metadatas.append({
            "path": path,
            "section_title": title,
            "start_page": actual_start_page + 1
        })
        
        safe_id = "".join(c if c.isalnum() else "_" for c in title).lower()
        ids.append(f"chunk_{original_toc_idx}_{safe_id[:15]}")
    
    print("-" * 50)

# --- FINAL SUMMARY REPORT ---
print("\n" + "="*50)
print("CHUNKING EXECUTION SUMMARY")
print("="*50)
print(f"Total Chunks Generated : {len(chunks)}")
print(f"Successful Boundary Cuts: {success_count}")
print(f"Failed Cuts: {debug_count}")

if failed_cases:
    print("\n--- LIST OF DEBUG / FAILED BOUNDARY CASES ---")
    for case in failed_cases:
        print(f"• Section {case['section_index']}: '{case['current_title']}'")
        print(f"  Target Next Title Missing: '{case['target_next_title']}'")
        print(f"  Characters Collected     : {case['collected_chars']}")
print("="*50)

Found 147 level 2 sections to process.
[1/147] Processing Section:
  • Looking for Start Title : 'Account Ownership'
  • Cutting Boundary Target : 'Pay On Death (POD) account'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[2/147] Processing Section:
  • Looking for Start Title : 'Pay On Death (POD) account'
  • Cutting Boundary Target : 'Accounts established for minors'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[3/147] Processing Section:
  • Looking for Start Title : 'Accounts established for minors'
  • Cutting Boundary Target : 'Transfer of account ownership'
  -> Found start anchor on Page 4
  -> [SUCCESS] Cleanly cut right before next title boundary.
--------------------------------------------------
[4/147] Processing Section:
  • Looking for Start Title : 'Transfer of account ow

In [34]:
target_chunk_idx = 32

if target_chunk_idx < len(chunks):
    print(f"Chunk {target_chunk_idx}")
    print("=" * 40)
    print("METADATA")
    print("-" * 40)
    for key, value in metadatas[target_chunk_idx].items():
        print(f"  {key}: {value}")
    print("\nTEXT CONTENT")
    print("-" * 40)
    print(chunks[target_chunk_idx])
    print("=" * 40)
    print(f"Character count: {len(chunks[target_chunk_idx])}")
else:
    print(f"Index {target_chunk_idx} is out of range. Total chunks available: {len(chunks)}")

Chunk 32
METADATA
----------------------------------------
  path: Account Basics > Availability of Funds Policy > Holds on other funds
  section_title: Holds on other funds
  start_page: 12

TEXT CONTENT
----------------------------------------
Holds on other funds
• The first $6,725 of a day’s total deposits of cashier’s, certified, teller’s, traveler’s, and federal, state, and 
local government checks, and U.S. Postal Service money orders made payable to you will be available 
on the first business day after the day of your deposit, if your deposit meets certain conditions. For 
example, the checks must be payable to you. If your deposit of these checks (other than U.S. Treasury 
checks) is not made in person to one of our employees, the first $6,725 may not be available until the 
second business day after the day of your deposit. 
• The excess over $6,725 and funds from all other check deposits will be available no later than the
seventh business day after the day of your deposit.

In [35]:
for i in range(len(chunks)):
    print(f"Chunk {i}: {len(chunks[i])} characters")

Chunk 0: 17 characters
Chunk 1: 28 characters
Chunk 2: 4696 characters
Chunk 3: 31 characters
Chunk 4: 53 characters
Chunk 5: 74 characters
Chunk 6: 18 characters
Chunk 7: 4625 characters
Chunk 8: 88 characters
Chunk 9: 43 characters
Chunk 10: 59 characters
Chunk 11: 3613 characters
Chunk 12: 2207 characters
Chunk 13: 1232 characters
Chunk 14: 22 characters
Chunk 15: 32 characters
Chunk 16: 31 characters
Chunk 17: 43 characters
Chunk 18: 38 characters
Chunk 19: 3341 characters
Chunk 20: 17 characters
Chunk 21: 32 characters
Chunk 22: 39 characters
Chunk 23: 46 characters
Chunk 24: 35 characters
Chunk 25: 4830 characters
Chunk 26: 4797 characters
Chunk 27: 5025 characters
Chunk 28: 32 characters
Chunk 29: 2110 characters
Chunk 30: 25 characters
Chunk 31: 2512 characters
Chunk 32: 1741 characters
Chunk 33: 1878 characters
Chunk 34: 2458 characters
Chunk 35: 3527 characters
Chunk 36: 143 characters
Chunk 37: 1199 characters
Chunk 38: 49 characters
Chunk 39: 4496 characters
Chunk 40: 63 ch

In [36]:
# 1. Initialize ChromaDB and create a unique collection for this document
chroma_client = chromadb.PersistentClient(path="./chroma_db")

doc_name = "wellsfargo_deposit_agreement"  # Replace with your dynamic document filename/identifier
collection_name = f"doc_{"".join(c if c.isalnum() else "_" for c in doc_name).lower()}"
collection = chroma_client.get_or_create_collection(name=collection_name)

print(f"Initialized unique ChromaDB collection: '{collection_name}'\n" + "="*50)

# 2. Configure the text splitter (500 chars, 100 overlap)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

final_documents = []
final_metadatas = []
final_ids = []

MAX_CHUNK_SIZE = 500  # Threshold: only sub-chunk if text exceeds this length

# 3. Process existing TOC chunks conditionally
for parent_idx, (section_text, parent_meta, parent_id) in enumerate(zip(chunks, metadatas, ids)):
    
    # Check if the section is too large and needs further splitting
    if len(section_text) > MAX_CHUNK_SIZE:
        splits = text_splitter.split_text(section_text)
        
        for sub_idx, split_text in enumerate(splits):
            sub_id = f"{parent_id}_sub_{sub_idx}"
            sub_meta = {
                "section_title": parent_meta.get("section_title", "Unknown"),
                "start_page": parent_meta.get("start_page", 1),
                "parent_chunk_id": parent_id,
                "is_sub_chunk": True,
                "sub_index": sub_idx
            }
            final_documents.append(split_text)
            final_metadatas.append(sub_meta)
            final_ids.append(sub_id)
    else:
        # If it's already small enough, keep the original TOC chunk whole
        parent_meta["is_sub_chunk"] = False
        final_documents.append(section_text)
        final_metadatas.append(parent_meta)
        final_ids.append(parent_id)

print(f"Processed {len(chunks)} TOC sections into {len(final_documents)} final storable chunks.")

# 4. Add everything to ChromaDB
if final_documents:
    collection.add(
        documents=final_documents,
        metadatas=final_metadatas,
        ids=final_ids
    )
    print(f"Successfully embedded and stored {len(final_documents)} chunks in ChromaDB!")
else:
    print("[WARNING] No documents found to store.")

Initialized unique ChromaDB collection: 'doc_wellsfargo_deposit_agreement'
Processed 147 TOC sections into 537 final storable chunks.
Successfully embedded and stored 537 chunks in ChromaDB!


# Citi

In [37]:
pdf_path = "citi-Consumer_Client_Manual.pdf"
pdf = pypdfium2.PdfDocument(pdf_path)

toc_page_index = 1
toc_text = pdf[1].get_textpage().get_text_range()

# String followed by space and page number
pattern = re.compile(r"([A-Za-z0-9\s,\-\(\):\.]+?)\s+(\d{1,3})\s*$", re.MULTILINE)

raw_toc_items = []
for match in pattern.finditer(toc_text):
    title = match.group(1).strip()
    page_num = int(match.group(2))
    if len(title) > 80:
        continue
    raw_toc_items.append({
        "title": title,
        "target_page": page_num,
    })

toc_sections = sorted(raw_toc_items, key=lambda x: x["target_page"])

print(f"\nGenerated {len(toc_sections)} sections:")
for item in toc_sections:
    print(f"Page {item['target_page']}: {item['title']}")


Generated 14 sections:
Page 3: 2 Simplified banking
Page 6: 3 Relationship Tiers
Page 10: 4 Account Statements
Page 12: 5 Products
Page 29: 6 Opening and Closing Accounts
Page 38: 7 Account Transactions
Page 49: 8 Electronic Fund Transfers
Page 60: 9 Other Fund Transfer Services
Page 64: 10 Additional Important Terms 
and Information
Page 66: 11 Arbitration
Page 69: Appendix 1: Fee Schedule
Page 72: Appendix 2: Funds Availability at Citibank
Page 75: Appendix 3: Discontinued Products, 
Services and Features
Page 77: Appendix 4: Definitions


In [38]:
target_page_idx =  5
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = textpage.get_text_range()

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 5
----------------------------------------
  1: 3
  2: 2 Simplified banking 
  3: With simplified banking, your consumer Eligible Deposit and Investment (EDI) accounts will be linked to create one 
  4: Combined Average Monthly Balance (CAMB) to determine your features and benefits. Higher CAMBs for deposit 
  5: account owners may allow you to join Relationship Tiers and receive Monthly Service Fee and Non-Citi ATM fee 
  6: waivers. Please refer to section 3 Relationship Tiers and section 5 Products. Simplified banking customers who own 
  7: multiple deposit accounts also may receive one Account Statement as explained in section 4 Account Statements. 
  8: Your CAMB is the total of the End of Day Available Now balances for all your EDI account(s) in a calendar month 
  9: divided by the number of days in that month. Your CAMB is based on the calendar month which may be different 
 10: than your Statement Period. You must be the owner (or beneficial owner) of an EDI account for 

In [39]:
def clean_usb(text):
    text = re.sub(r"^\s*\d{1,3}\s*\n", "", text) # Remove page numbers at the start of the text
    return text

target_page_idx =  62
page = pdf[target_page_idx - 1]

textpage = page.get_textpage()
page_text = clean_usb(textpage.get_text_range())

print(f"Page {target_page_idx}")
print("-" * 40)
lines = page_text.splitlines()

for idx, line in enumerate(lines, start=1):
    if line.strip():
        print(f"{idx:3d}: {line}")

Page 62
----------------------------------------
  1: 9 Other Fund Transfer Services
  2: 9.1 Outgoing Wire Transfers 
  3: The following terms apply to any wire transfer requests we receive to transfer funds from your account to an 
  4: account maintained at another institution. In addition to the information provided here, please refer to the 
  5: separate funds transfer agreement governing the terms of service for transfers made using Citi Online or the Citi 
  6: Mobile App. Please note that outgoing international transfers from a consumer account are also governed by 
  7: special provisions regarding disclosures you receive and error resolution rights as more fully described in section 
  8: 9.2 Remittance Transfers of this Agreement. Before utilizing our wire transfer services, please read the following 
  9: important tips for wire transfer safety.
 10: General Tips for Wire Transfer Safety 
 11: Wire transfers are a fast way to send money to individuals and businesses. Howev

In [40]:
pages = []
for i in range(total_pages):
    textpage = pdf[i].get_textpage()
    cleaned_text = clean_usb(textpage.get_text_range())
    pages.append(cleaned_text)